In [ ]:
import torch
import sounddevice as sd
import numpy as np
from scipy.io.wavfile import write
from pathlib import Path
import subprocess
import time
from transformers import AutoTokenizer
import onnxruntime as ort
from ollama import chat
from kokoro import KPipeline
from IPython.display import Audio, display
import soundfile as sf

# ============================================================
# Configuration
# ============================================================

SAMPLE_RATE = 16000
CHUNK_SIZE = 512

SPEECH_THRESHOLD = 0.5
SILENCE_DURATION = 1.0
MAX_UTTERANCE_SECONDS = 30

LLM_MODEL = "qwen2.5-coder:3b"

WHISPER_DIR = Path("../whisper.cpp")
WHISPER_MODEL = WHISPER_DIR / "models" / "ggml-base.en.bin"
WHISPER_CLI = WHISPER_DIR / "build" / "bin" / "whisper-cli"

AUDIO_FILE = Path("utterance.wav")

SHOULD_RESPOND_CLASS = 1

VOICE = "af_heart"

SYSTEM_PROMPT = """Understand the user's intent and context, and determine whether they need help, advice, suggestions, or an explanation.
Do not respond to every statement; stay silent when the user does not need assistance.
Proactively respond when the user is confused, stuck, asking a question, making a decision, or would clearly benefit from your help.
Act like an intelligent mentor: be concise, practical, context-aware, and respond only when your intervention is useful. and try responding in 4 to 5 lines"""


# ============================================================
# Load Should AI Respond model
# ============================================================

model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True
)

tokenizer = AutoTokenizer.from_pretrained(
    "./should_ai_respond_model"
)

print("Loaded tokenizer")


int8_session = ort.InferenceSession(
    "./should_ai_respond_int8.onnx",
    providers=["CPUExecutionProvider"]
)

print("Loaded INT8 BERT classification model")

pipeline = KPipeline(lang_code="a")

stream = sd.OutputStream(
    samplerate=24000,
    channels=1,
    dtype="float32",
    blocksize=2048
)


# ============================================================
# Helper
# ============================================================

def softmax(x):
    exp_x = np.exp(
        x - np.max(x, axis=1, keepdims=True)
    )
    return exp_x / exp_x.sum(
        axis=1,
        keepdims=True
    )


# ============================================================
# Main loop
# ============================================================

print("Listening...")
print("Speak now.\n")


while True:

    # Reset state for every utterance

    audio_chunks = []
    speech_started = False
    silence_start = None

    utterance_start_time = time.time()

    model.reset_states()

    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32",
        blocksize=CHUNK_SIZE
    ) as stream:

        while True:

            chunk, overflowed = stream.read(CHUNK_SIZE)

            chunk = chunk[:, 0]

            chunk_tensor = torch.from_numpy(chunk)

            speech_probability = model(
                chunk_tensor,
                SAMPLE_RATE
            ).item()

            is_speech = (
                speech_probability >= SPEECH_THRESHOLD
            )

            # ----------------------------------------
            # Speech
            # ----------------------------------------

            if is_speech:

                if not speech_started:
                    print("Speech detected...")
                    speech_started = True

                audio_chunks.append(chunk.copy())

                silence_start = None

            # ----------------------------------------
            # Silence
            # ----------------------------------------

            else:

                if speech_started:

                    audio_chunks.append(chunk.copy())

                    if silence_start is None:
                        silence_start = time.time()

                    silence_time = (
                        time.time() - silence_start
                    )

                    if silence_time >= SILENCE_DURATION:
                        print("User finished speaking.")
                        break

            # ----------------------------------------
            # Maximum utterance duration
            # ----------------------------------------

            if (
                time.time() - utterance_start_time
                >= MAX_UTTERANCE_SECONDS
            ):
                print("Maximum utterance duration reached.")
                break


    # ========================================================
    # No speech
    # ========================================================

    if not speech_started:
        print("No speech detected.")
        continue


    # ========================================================
    # Combine audio
    # ========================================================

    audio = np.concatenate(audio_chunks)

    write(
        AUDIO_FILE,
        SAMPLE_RATE,
        audio
    )


    # ========================================================
    # Whisper
    # ========================================================

    print("Calling Whisper...")

    try:

        result = subprocess.run(
            [
                str(WHISPER_CLI),
                "-m", str(WHISPER_MODEL),
                "-f", str(AUDIO_FILE),
                "-nt"
            ],
            capture_output=True,
            text=True,
            check=True
        )

    except subprocess.CalledProcessError as e:

        print("Whisper failed:")
        print(e.stderr)

        continue


    user_text = result.stdout.strip()

    if not user_text:
        print("Whisper returned empty text.")
        continue


    print("\nUser:")
    print(user_text)


    # ========================================================
    # BERT - Should AI Respond?
    # ========================================================

    inputs = tokenizer(
        user_text,
        return_tensors="np",
        truncation=True
    )

    onnx_inputs = {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"]
    }

    logits = int8_session.run(
        None,
        onnx_inputs
    )[0]

    probs = softmax(logits)

    prediction = np.argmax(
        probs,
        axis=1
    )[0]

    respond_probability = probs[0][SHOULD_RESPOND_CLASS]

    print(
        f"Should respond probability: "
        f"{respond_probability:.3f}"
    )


    # ========================================================
    # Decision
    # ========================================================

    if prediction != SHOULD_RESPOND_CLASS:

        print("BERT decided: DON'T RESPOND")
        continue


    print("BERT decided: RESPOND")


    # ========================================================
    # LLM
    # ========================================================

    stream = chat(
        model=LLM_MODEL,
        messages=[
            {
                "role":"system",
                "content":"You are helpfull AI Assistent, for given text or question you will respond in short like normal person, give long response only if asked or needed"
            },
            {
                "role": "user",
                "content": user_text
            }
            ],
            stream=True
    )

    print("\nLLM:")
    audio_streamer = sd.OutputStream(
            samplerate=24000,
            channels=1,
            dtype="float32",
            blocksize=2048
        )
    audio_streamer.start()
    try:
        for chunk in stream:
            text = chunk["message"]["content"]
            print(text, end="", flush=True)
    
            generator = pipeline(text, voice=VOICE)
    
            for _, _, audio in generator:
                audio_streamer.write(audio)

    finally:
        audio_streamer.stop()     
        audio_streamer.close()
        
    

### observation

1. Pipelie is working all the compenents are working fine and can here the output
2. The output is not smooth, it is taking some time to output audio word

### Next Step

1. **Multithreading and shared memory**